<a href="https://colab.research.google.com/github/Lateephah/Applied-Search-Intelligence-System/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Lateephah/Applied-Search-Intelligence-System/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# My Response:
Ranking. My task is to rank content pages by their priority for review, so content and SEO teams can focus limited editing time on the pages most worth investigating first. Ranking fits the decision better than simple classification because the practical question is not only whether a page is declining, but which pages should be reviewed first. I will test whether combining multiple observable content, search, and performance signals can improve the prioritization of review candidates.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Lateephah/Applied-Search-Intelligence-System"
REPO_DIR = "Applied-Search-Intelligence-System"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

In [ ]:
from pathlib import Path
path = Path("scripts/01_prepare_features.py")
print(path.read_text())

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from ml_utils import (
    BOOLEAN_COLUMNS,
    CATEGORICAL_COLUMNS,
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    NUMERIC_COLUMNS,
    PROCESSED_DIR,
    RAW_PATH,
    display_path,
    ensure_dirs,
    to_bool_series,
    write_json,
)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Prepare FlyRank refresh feature vector.")
    parser.add_argument("--input", default=str(RAW_PATH), help="Raw anonymized CSV export.")
    parser.add_argument(
        "--output",
        default=str(PROCESSED_DIR / "refresh_feature_vector.csv"),
        help="Prepared feature-vector CSV.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    ensure_dirs()

    input_path = Path(args.input)
    if not input_path.exists():
        raise FileNotFoundError(
            f"Raw input not found: {input

# My Reponse:
The starter pipeline creates `is_declining_label` from `trend_direction`, where `"down"` is assigned 1 and all other directions are assigned 0. This is a **rule-derived proxy**, not an independently observed future outcome. I will use it provisionally to test whether a model can prioritize pages associated with the existing declining-page definition. This means the model can support prioritization of pages showing the defined decline pattern, but it cannot establish that a page will decline in the future or that refreshing it will prevent decline. I will exclude `trend_direction` and `trend_pct` from the model features because they are used to define the proxy target.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

# My Reponse:
Precision@50 is my primary success metric because the practical decision is to identify a small number of pages for review first. It measures the proportion of the top 50 ranked pages that match the target outcome. A good ranking should improve Precision@50 compared with the transparent baseline, meaning more relevant review candidates appear in the first 50 pages.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

 I originally loaded `outputs/refresh_queue_sample.csv` here. Checking `skills/flyrank/flyrank-data/SKILL.md`, that file isn't one of the project's two real datasets (the starter CSV or the warehouse) -- it's a 200-row worked example of a *finished* pipeline output (it already has `final_rank`, `best_model_name`, `best_model_probability` columns), shipped for illustration. It was never my lane's actual population.

I load my lane's real slice from `data/raw/content_refresh_anonymized.csv` instead -- the starter dataset the skill names, applying the same two filters the starter pipeline (`scripts/01_prepare_features.py`) uses: `impressions_90d > 0` (the page has real search visibility to judge) and `content_age_days >= 90` (the page is old enough to have a real 90-day trend, which is what `trend_direction`/`is_declining_label` are built from).

In [ ]:
import pandas as pd

raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")
initial_rows = len(raw)

# Same lane filter the starter pipeline applies, real search visibility, and old
# enough to have an actual 90-day trend to be "declining" from.
lane = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
lane = lane.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Same proxy-target definition from cell above: NEVER an observed future outcome,
# a rule applied to trend_direction (itself derived from trend_pct).
lane["is_declining_label"] = lane["trend_direction"].str.lower().eq("down").astype(int)

print(f"Rows: {len(lane):,} of {initial_rows:,} raw rows")
print(f"Unique content IDs: {lane['content_id'].nunique():,}")
print(f"Declining rate (share of is_declining_label == 1): {lane['is_declining_label'].mean():.3f}")

lane[[
    "content_id",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "impressions_90d",
    "is_declining_label",
]].head()

Rows: 30,000 of 30,000 raw rows
Unique content IDs: 30,000
Declining rate (share of is_declining_label == 1): 0.542


,content_id,content_age_days,days_since_last_update,avg_position,impressions_90d,is_declining_label
0,content_304f48230142,187,20,10.6,3803,1
1,content_a1fb4e703a9e,445,25,20.3,15320,1
2,content_9aa793d4d895,141,20,36.5,12581,1
3,content_331d6c4de07b,463,22,6.2,11751,0
4,content_d99b7a2d90ca,263,14,44.0,19140,1


One row = one content page. My lane's real population is 30,000 pseudonymized content items (one per unique `content_id`) in the starter export; after the visibility and age filter, all 30,000 rows qualify (every row already has impressions and is old enough), so the working slice is the full starter set. That's a very different shape than the 200-row demo file I'd mistakenly used before -- and it means my base rate, and later my Precision@50 baseline, are measured on the real population size, not a small illustrative sample.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

#My Response:
A fixed rule such as prioritizing pages based only on age or freshness may miss other signals that indicate a page is worth reviewing. The dataset contains multiple content, search, engagement, and performance signals that may interact, making a single threshold potentially too simple. ML is worth testing if it can combine these signals and improve Precision@50 over a transparent baseline. If it does not improve the baseline, the simpler rule may be the better solution.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.